In [1]:
# %load_ext autoreload
# %autoreload 2

# The Barchart settle fetcher underneath `sfr_cvx_adj` calls `asyncio.run()`, and
# a Jupyter kernel already owns a running loop -- without this the whole notebook
# dies on "asyncio.run() cannot be called from a running event loop". This is why
# eod_linear_rates.ipynb opens the same way.
import nest_asyncio
nest_asyncio.apply()

import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use("ggplot")
pylab.rcParams.update({
    "legend.fontsize": "medium", "figure.figsize": (18, 6),
    "axes.labelsize": "medium", "axes.titlesize": "medium",
    "xtick.labelsize": "medium", "ytick.labelsize": "medium",
})

import datetime
import os
import sys

import numpy as np
import pandas as pd
import pytz

os.environ.setdefault("ARBS_SUPABASE_ENABLED", "0")
NYC_tz = pytz.timezone("America/New_York")
sys.path.append("../../")

from RVUtils.plt_timeseries import make_secondary_axis_plot

# SFR convexity adjustment

`CA = pack_rate − matched_swap_rate`, in basis points, for each of the five
pack colours Citi prints. The pack rate is the average of four consecutive SR3
settles; the matched swap is the forward swap covering the same window,
**quarterly/quarterly** — Citi specifies that verbatim, and the `usd_irs` spec
quotes annual fixed, which is a 4.6–5.9 bp error on a quantity that is itself
1–20 bp.

Two things to know before reading any number here:

* **The adjustment grows with rank.** Whites sit ~0.3 months out and Golds
  ~4 years; `CA = ½·σ²·mean(T1²)`, so it is a variance quantity and the deeper
  colour should always carry more. That monotonicity is the cheapest sanity
  check available on this series.
* **Deep colours were unavailable until recently.** Blues had 49 usable dates
  in 2023 and Golds had none at all in 2026 before the SR3 settle warm; they
  now carry ~250 dates a year.

In [22]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.IRSwapsTB import IRSwapsTB
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP, _alias_to_cusip
from MDP.IRSwaptions.IRSwaptionMDP import IRSwaptionMDP
from TB.IRSwaptionsTB import IRSwaptionsTB
from TB.FixedRateBondsTB import FixedRateBondsTB
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedStructure, UnifiedValue
from TB.TimeseriesBuilder import TimeseriesBuilder

# curve_mdp = IRSwapsMDP(source="CITIVELO_EXCEL")
curve_mdp = IRSwapsMDP(source="citivelo_excel_rl")
usts_mdp = FixedRateBondsMDP(source="USTS_FEDINVEST_WSJ_LIVE-RL")
vol_mdp = IRSwaptionMDP(
    source="CITIVELO-RL",
    curve_source="citivelo_excel_rl",
    request_defaults={"verify": False},
)

start = datetime.date(2022, 1, 2)
end = datetime.date(2026, 8, 20)

In [23]:
COLOURS = ["WHITES", "REDS", "GREENS", "BLUES", "GOLDS"]
ca_df = IRSwapsTB(curve_mdp, show_tqdm=True).sfr_cvx_adj(COLOURS, start, end)
ca_df


FETCHING EOD DATA FROM BARCHART...: 100%|██████████| 28/28 [00:01<00:00, 19.39it/s]


,USD-SOFR-1D WHITES PACKS CVX_ADJ,USD-SOFR-1D REDS PACKS CVX_ADJ,USD-SOFR-1D GREENS PACKS CVX_ADJ,USD-SOFR-1D BLUES PACKS CVX_ADJ,USD-SOFR-1D GOLDS PACKS CVX_ADJ
Date,,,,,
2022-01-03,0.217280,1.690529,5.180350,7.591631,13.413164
2022-01-04,0.966147,0.473140,6.020419,9.295477,12.998421
2022-01-05,1.489321,0.107497,5.396419,7.262716,11.599422
2022-01-06,0.929450,0.815511,5.214086,6.840175,11.385335
2022-01-07,0.109376,1.464996,5.687598,7.553039,11.807058
...,...,...,...,...,...
2026-08-14,0.434839,0.070941,2.755935,6.869951,9.965899
2026-08-17,0.066234,0.562448,2.519989,5.684035,8.738245
2026-08-18,0.186018,0.787803,2.620597,6.171205,9.484143


In [ ]:
df = TimeseriesBuilder().get_timeseries(
    start=start,
    end=end,
    queries=[
        UnifiedQuery(
            curve="USD-SOFR-1D",
            tenor="2y/5y/10y",
            value=UnifiedValue.IRS_RATE,
        ),
		UnifiedQuery(
            curve="USD-SOFR-1D",
            tenor="5y/10y/30y",
            value=UnifiedValue.IRS_RATE,
        ),
		UnifiedQuery(
			curve="USD-SOFR-1D",
			selector={"shorthand":  "1Yx10Y", "strike": "ATMF"},
			structure=UnifiedStructure.IRSWAPTION_STRADDLE,
			value=UnifiedValue.IRSWAPTION_NVOL,
		)
    ],
    n_jobs=12,
    routers={
        "IRS": IRSwapsTB(curve_mdp, show_tqdm=True),
        "FRB": FixedRateBondsTB(usts_mdp, show_tqdm=True),
        "IRSWAPTION":  IRSwaptionsTB(vol_mdp, show_tqdm=True),
    },
)
df

PRICING IRSWAPTIONS.:   0%|          | 0/1 [00:00<?, ?it/s]

SUCCESS: `func_tol` reached after 6 iterations (levenberg_marquardt), `f_val`: 2.9176159133483135e-13, `time`: 0.2212s
SUCCESS: `func_tol` reached after 6 iterations (levenberg_marquardt), `f_val`: 3.495995652227006e-13, `time`: 0.2297s


WARNING	Thread(ts-builder_1) MDP.IRSwaptions.CITIVELO.provider:provider.py:_cube_for_date()- Citi vol for 2022-01-17 is not in the swaption cube store and no client was supplied, so this will connect to Excel over COM. Warm it with scripts/citivelo_swaption_vol_warm.py build, or pass use_cube_store=False if driving Excel is what you meant.


SUCCESS: `func_tol` reached after 6 iterations (levenberg_marquardt), `f_val`: 2.7171190269755225e-13, `time`: 0.1710s
SUCCESS: `func_tol` reached after 6 iterations (levenberg_marquardt), `f_val`: 2.9176159133483135e-13, `time`: 0.1750s
SUCCESS: `func_tol` reached after 6 iterations (levenberg_marquardt), `f_val`: 3.495995652227006e-13, `time`: 0.1735s


WARNING	Thread(ts-builder_0) IRSwapsTB:IRSwapsTB.py:get_timeseries()- citivelo_excel_rl: no curve for 1 of 1154 requested point(s) on 'USD-SOFR-1D' — those rows are ABSENT from the result, not NaN (first=2026-08-20, last=2026-08-20).
WARNING	Thread(ts-builder_1) MDP.CitiVelocityExcel.vol.cube_data:cube_data.py:_drop_incomplete()- SwaptionCubeData: dropped 8 expiry(ies) ['9M', '1Y', '18M', '2Y', '10Y', '12Y', '15Y', '20Y'] and 2 tenor(s) ['4Y', '12Y'] with incomplete quotes (strict=False). Nothing was NaN-filled.
WARNING	Thread(ts-builder_1) MDP.IRSwaptions.CITIVELO.provider:provider.py:_cube_for_date()- Citi vol for 2022-02-21 is not in the swaption cube store and no client was supplied, so this will connect to Excel over COM. Warm it with scripts/citivelo_swaption_vol_warm.py build, or pass use_cube_store=False if driving Excel is what you meant.
WARNING	Thread(ts-builder_1) MDP.CitiVelocityExcel.vol.cube_data:cube_data.py:_drop_incomplete()- SwaptionCubeData: dropped 9 expiry(ies) ['

In [25]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(
    engine="plotly", 
	# ylabel_left="convexity adjustment, bp",
    # title="SFR convexity adjustment by pack colour"
)

plot(ca_df["USD-SOFR-1D BLUES PACKS CVX_ADJ"].dropna(), which="left", label=c)
legend(show_date=True)

## The current term structure

In [6]:
latest = ca.dropna(how="all").iloc[-1]
asof = ca.dropna(how="all").index[-1]
print(f"as of {asof.date()}")
print(latest.reindex(COLOURS).round(3).to_string())

import plotly.graph_objects as go
_c = latest.reindex(COLOURS).dropna()
f2 = go.Figure(go.Bar(x=_c.index, y=_c.to_numpy(),
                      text=[f"{v:.2f}" for v in _c], textposition="outside"))
f2.update_layout(title=f"SFR convexity adjustment by colour, {asof.date()}",
                 yaxis_title="bp", height=380)
f2.show()

as of 2026-08-20
WHITES    0.350
REDS      0.560
GREENS    2.624
BLUES     6.648
GOLDS     9.934


## Against the 2s5s10s fly

Citi's Figure 9 argues the adjustment is directional with the 2s5s10s swap
fly, and therefore hedgeable with it. On 2021–2026 SOFR that relationship did
**not** reproduce — hedge R² 0.000–0.010 with a sign-flipping beta — so this
panel is here to be looked at rather than relied on.

In [7]:
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapStructure import IRSwapStructure
from Query.IRSwaps.IRSwapValue import IRSwapValue

try:
    # The slash form is what the query layer auto-detects as a FLY; passing
    # `structure_kwargs={"tenors": [...]}` raises "FLY requires all three leg
    # tenors". Construction is inside the try because that is where it raises.
    fly_q = IRSwapQuery(curve="USD-SOFR-1D", tenor="2Y/5Y/10Y",
                        value=IRSwapValue.RATE, structure_kwargs={"bpv": 1.0})
    fly = tb.get_timeseries(start=start, end=end, queries=[fly_q], n_jobs=8)
    fly.columns = ["2s5s10s fly"]
    print(fly.tail(3).to_string())
except Exception as exc:                                     # noqa: BLE001
    fly = pd.DataFrame()
    print(f"fly unavailable: {type(exc).__name__}: {exc}")

SUCCESS: `func_tol` reached after 6 iterations (levenberg_marquardt), `f_val`: 3.163917978563504e-13, `time`: 0.1535s



PRICING USD-SOFR-1D IRSWAPS [workers=8]...:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\chris\anaconda3\envs\stir\Lib\site-packages\rateslib\data\fixings.py:3426: RuntimeWarning:

invalid value encountered in divide


                                                                                 

            2s5s10s fly
Date                   
2026-08-18      -14.241
2026-08-19      -13.179
2026-08-20      -12.036


In [8]:
if not fly.empty:
    j = ca.join(fly, how="inner").dropna(subset=["2s5s10s fly"])
    plot, fig3, ax, ax2, legend = make_secondary_axis_plot(
        engine="plotly", ylabel_left="CA, bp", ylabel_right="fly, bp",
        title="Greens / Blues convexity adjustment vs the 2s5s10s fly")
    for c in ("GREENS", "BLUES"):
        if c in j.columns and j[c].notna().any():
            plot(j[c].dropna(), which="left", label=c)
    plot(j["2s5s10s fly"].dropna(), which="right", label="2s5s10s fly")
    legend(show_date=True)
    fig3.show()

    corr = j[[c for c in COLOURS if c in j.columns]].corrwith(j["2s5s10s fly"])
    print("\ncorrelation of LEVELS with the fly:")
    print(corr.round(3).to_string())
    dcorr = j.diff().dropna()
    print("\ncorrelation of daily CHANGES with the fly (the one that matters):")
    print(dcorr[[c for c in COLOURS if c in j.columns]]
          .corrwith(dcorr["2s5s10s fly"]).round(3).to_string())
else:
    print("skipped: no fly series")


correlation of LEVELS with the fly:
WHITES    0.092
REDS      0.126
GREENS    0.375
BLUES     0.623
GOLDS     0.528

correlation of daily CHANGES with the fly (the one that matters):
WHITES    0.026
REDS     -0.028
GREENS   -0.014
BLUES    -0.096
GOLDS    -0.095


In [9]:
tb.close()
print("done")

done
